In [ ]:
!pip install requests beautifulsoup4 lxml tqdm ipywidgets

## Импорты

In [1]:
import requests
from bs4 import BeautifulSoup
import db
from config import BASE_URL, INDEX_URL, HEADERS
from tqdm.notebook import tqdm
import time
import concurrent.futures
import threading

In [2]:
db.init_db()

## Парсим индекс

In [3]:
response = requests.get(INDEX_URL, headers=HEADERS)
response.raise_for_status()

soup = BeautifulSoup(response.text, 'lxml')
items = soup.select_one('div.items2024').select('div.item2024')

In [ ]:
for item in tqdm(items, desc="Парсим товары"):
    name_tag = item.select_one("div.name a")
    name = name_tag.get_text(strip=True)
    link = name_tag["href"]
    price = item.select_one("div.price").get_text(strip=True)[:-2]

    db.upsert_product(name, price, link)

## Парсинг товаров

In [3]:
products = db.get_all_products()

In [4]:
len(products)

8259

### Парсинг категории

In [5]:
def parse_category(soup: BeautifulSoup) -> tuple[list[str], int | None]:
    crumbs = soup.select("ul.breadcrumbs2024 li")

    category_names = []
    for li in crumbs[1:-1]:
        span = li.select_one("[itemprop=name]")
        if span:
            category_names.append(span.get_text(strip=True))

    if not category_names:
        return [], None

    category_id = db.ensure_category_path(category_names)
    return category_id

### Парсинг описания и характеристик

In [6]:
def parse_description_and_characteristics(soup: BeautifulSoup, product_id: int):
    desc_block = soup.select_one(
        'div.good-tabs-body > div.description-body[itemprop="description"]'
    )

    desc_text = None
    if desc_block:
        paragraphs = [
            p.get_text(" ", strip=True)
            for p in desc_block.find_all("p", recursive=False)
        ]
        desc_text = "\n\n".join(p for p in paragraphs if p)
        db.add_product_description(
            product_id,
            desc_text
        )

        # блок характеристик может быть внутри описания
        ul = desc_block.find("ul")
        if ul:
            for li in ul.find_all("li"):
                text = li.get_text(" ", strip=True)
                if ":" in text:
                    key, value = map(str.strip, text.split(":", 1))
                    if key and value:
                        db.add_product_characteristic(
                            product_id,
                            db.upsert_characteristic(key),
                            value,
                        )

        # weigths характеристики
        weights = desc_block.select_one("div.weights")
        if weights:
            for div in weights.find_all("div"):
                text = div.get_text(" ", strip=True)
                if ":" in text:
                    key, value = map(str.strip, text.split(":", 1))
                    if key and value:
                        db.add_product_characteristic(
                            product_id,
                            db.upsert_characteristic(key),
                            value,
                        )

        # таблица характеристик
        table = soup.select_one("div.good-tabs-body > div.description-body[data-column-id='2'] table.good-param")
        if table:
            for tr in table.find_all("tr"):
                tds = tr.find_all("td")

                if len(tds) != 2:
                    continue

                key = tds[0].get_text(" ", strip=True)
                value = tds[1].get_text(" ", strip=True)
                if not key or key.lower() == "характеристики":
                    continue

                if key and value:
                    db.add_product_characteristic(
                        product_id,
                        db.upsert_characteristic(key),
                        value,
                    )

    return desc_text

### Парсинг непосредственно

In [9]:
for url, product_id in tqdm(products, desc="Парсинг карточек"):
    try:
        response = requests.get(BASE_URL + url, headers=HEADERS)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'lxml')

        category_id = parse_category(soup)
        if category_id:
            db.update_product_category(product_id, category_id)
        parse_description_and_characteristics(soup, product_id)
        time.sleep(0.2)
    except Exception as e:
        print(f"Ошибка на {url}: {e}")

Парсинг карточек:   0%|          | 0/8259 [00:00<?, ?it/s]

Ошибка на /goods/akrilovaya-elochnaya-igrushka-shishka-s-hrustalnoj-vetvi-8-sm-serebryanaya-podveska/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/angel-belosnezhnyj-s-muzykalnymi-instrumentami-na-podveske-10-sm/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/angelochek-v-belom-11-sm/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/angelochek-v-belom-na-olene-12-sm/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/babochka-zhemchuzhnaya-belaya-20h18-sm-klipsa/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/bant-azhurnyj-s-kamnem-serebryanyj-12x13-sm-/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/bant-belyj-s-serebryanoj-otdelkoj-23x33-sm10758/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/bant-fantaziya-iz-dzhuta-belyj-28x4x34-sm/: module 'db' has no attribute 'update_product_category'
Ошибка на /goods/ba

KeyboardInterrupt: 

## Суперпарсинг + бан

In [7]:
db_lock = threading.Lock()

def process_product(item):
    url, product_id = item

    try:
        response = requests.get(BASE_URL + url, headers=HEADERS, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'lxml')

        category_id = parse_category(soup)

        with db_lock:
            if category_id:
                db.update_product_category(product_id, category_id)

            parse_description_and_characteristics(soup, product_id)

        time.sleep(0.5)
        return True

    except Exception as e:
        print(f"Ошибка на {url}: {e}")
        return False


workers_count = 5

products = db.get_all_products()

with concurrent.futures.ThreadPoolExecutor(max_workers=workers_count) as executor:
    results = list(tqdm(executor.map(process_product, products), total=len(products), desc="Параллельный парсинг"))

Параллельный парсинг:   0%|          | 0/8259 [00:00<?, ?it/s]

Ошибка на /goods/elochnaya-igrushka-oss-rozhdestvenskij-brabant-14-sm-podveska/: database is locked
Ошибка на /goods/elochnoe-ukrashenie-girlyanda-zolotistaya-steklo/: database is locked
Ошибка на /goods/elochnoe-ukrashenie-hram---white-lace-9-sm-podveska/: database is locked
Ошибка на /goods/nabor-elochnyh-igrushek-skazochnyj-les---rainbow-9-15-sm-5-sht-podveska/: 404 Client Error: Not Found for url: https://www.eli.ru/goods/nabor-elochnyh-igrushek-skazochnyj-les---rainbow-9-15-sm-5-sht-podveska/
Ошибка на /goods/nabor-steklyannyh-sharov-la-ballare-8-sm-12-sht/: database is locked
Ошибка на /goods/puansettiya-giovanna-25-sm-belaya-na-steble/: HTTPSConnectionPool(host='www.eli.ru', port=443): Max retries exceeded with url: /goods/puansettiya-giovanna-25-sm-belaya-na-steble/ (Caused by SSLError(SSLEOFError(8, '[SSL: UNEXPECTED_EOF_WHILE_READING] EOF occurred in violation of protocol (_ssl.c:1032)')))
Ошибка на /goods/puansettiya-giovanna-25-sm-serebryanaya-na-steble/: HTTPSConnectionPoo